In [72]:
#Loading in Packages and Data

#Importing Packages
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.ticker as ticker
import matplotlib.cm as cm
from matplotlib.colors import Normalize
from matplotlib.ticker import MaxNLocator
from matplotlib.ticker import ScalarFormatter
import matplotlib.gridspec as gridspec
import xarray as xr
import os; import time
import pickle
import h5py
###############################################################
def coefs(coefficients,degree):
    coef=coefficients
    coefs=""
    for n in range(degree, -1, -1):
        string=f"({coefficients[len(coef)-(n+1)]:.1e})"
        coefs+=string + f"x^{n}"
        if n != 0:
            coefs+=" + "
    return coefs
###############################################################

# Importing Model Data
check=False
dir='/mnt/lustre/koa/koastore/torri_group/air_directory/DCI-Project/'

# # dx = 1 km; Np = 1M; Nt = 5 min
# data1=xr.open_dataset(dir+'../cm1r20.3/run/cm1out_1km_5min.nc') #***
# parcel1=xr.open_dataset(dir+'../cm1r20.3/run/cm1out_pdata_1km_5min_1e6.nc') #***
# res='1km';t_res='5min'
# Np_str='1e6'

# # dx = 1km; Np = 50M
# #Importing Model Data
# dir2='/home/air673/koa_scratch/'
# data1=xr.open_dataset(dir2+'cm1out_1km_1min.nc') #***
# parcel1=xr.open_dataset(dir2+'cm1out_pdata_1km_1min_50M.nc') #***
# res='1km'; t_res='1min'; Np_str='50e6'

# # dx = 1km; Np = 50M; Nz = 95
# #Importing Model Data
# dir2='/home/air673/koa_scratch/'
# data1=xr.open_dataset(dir2+'cm1out_1km_1min_95nz.nc') #***
# parcel1=xr.open_dataset(dir2+'cm1out_pdata_1km_1min_95nz.nc') #***
# res='1km'; t_res='1min_95nz'; Np_str='50e6'

# dx = 250m; Np = 50M
#Importing Model Data
dir2='/home/air673/koa_scratch/'
data1=xr.open_dataset(dir2+'cm1out_250m_1min_50M.nc') #***
parcel1=xr.open_dataset(dir2+'cm1out_pdata_250m_1min_50M.nc') #***
res='250m'; t_res='1min'; Np_str='50e6'

In [73]:
w_thresh1=0.1
w_thresh2=0.5
qcqi_thresh=1e-6

In [74]:
%%time

shape = data1['winterp'].shape  # (nx, ny, nz, nt)
# chunk_sizes = (256*2, 100, 10, 25)  # chunk x, chunk y, chunk z, chunk time
chunk_sizes = (256*2, 256*2, 10, 25*2)  # chunk x, chunk y, chunk z, chunk time

# Estimate total number of chunks
nx_chunks = (shape[0] + chunk_sizes[0] - 1) // chunk_sizes[0]
ny_chunks = (shape[1] + chunk_sizes[1] - 1) // chunk_sizes[1]
nz_chunks = (shape[2] + chunk_sizes[2] - 1) // chunk_sizes[2]
nt_chunks = (shape[3] + chunk_sizes[3] - 1) // chunk_sizes[3]
total_chunks = nx_chunks * ny_chunks * nz_chunks * nt_chunks

def GetUniqueCoords(t, y, x):
    coords = np.stack((t, y, x), axis=1)
    unique_coords = np.unique(coords, axis=0)
    return unique_coords[:, 0], unique_coords[:, 1], unique_coords[:, 2]

def chunk_all(shape, chunk_sizes):
    for xs in range(0, shape[0], chunk_sizes[0]):
        for ys in range(0, shape[1], chunk_sizes[1]):
            for zs in range(0, shape[2], chunk_sizes[2]):
                for ts in range(0, shape[3], chunk_sizes[3]):
                    yield (
                        slice(xs, min(xs + chunk_sizes[0], shape[0])),
                        slice(ys, min(ys + chunk_sizes[1], shape[1])),
                        slice(zs, min(zs + chunk_sizes[2], shape[2])),
                        slice(ts, min(ts + chunk_sizes[3], shape[3]))
                    )

lfc_profile = np.zeros((1, 2))

# Loop now chunks all dimensions
for i, (xs, ys, zs, ts) in enumerate(chunk_all(shape, chunk_sizes), 1):
    print(f'Processing chunk {i}/{total_chunks}: x={xs}, y={ys}, z={zs}, t={ts}')

    lfc_chunk = data1['lfc'].isel(xh=xs, yh=ys, time=ts).data
    w_chunk = data1['winterp'].isel(xh=xs, yh=ys, zh=zs, time=ts).data
    q_chunk = (data1['qc'].isel(xh=xs, yh=ys, zh=zs, time=ts).data +
               data1['qi'].isel(xh=xs, yh=ys, zh=zs, time=ts).data)

    # (x, y, z, t) mask
    mask = (w_chunk > w_thresh2) & (q_chunk > qcqi_thresh)
    t, z, y, x = np.where(mask)
    unique_coords = GetUniqueCoords(t, y, x)
    lfc_list = lfc_chunk[unique_coords[0], unique_coords[1], unique_coords[2]]
    lfc_list = lfc_list[lfc_list > 0]

    lfc_profile[:, 0] += np.sum(lfc_list)
    lfc_profile[:, 1] += len(lfc_list)


Processing chunk 1/13120: x=slice(0, 512, None), y=slice(0, 95, None), z=slice(0, 10, None), t=slice(0, 25, None)
Processing chunk 2/13120: x=slice(0, 512, None), y=slice(0, 95, None), z=slice(0, 10, None), t=slice(25, 50, None)



KeyboardInterrupt



In [28]:
#TAKING MEAN
mean_LFC = lfc_profile[:,0]/lfc_profile[:,1]
print(mean_LFC)

#OUTPUTTING
import pickle
dir2=dir+f'Project_Algorithms/Domain_Profiles/'
output_file=dir2+f'mean_lfc_{res}_{t_res}_{Np_str}.pkl'
with open(output_file, 'wb') as f:
    pickle.dump(mean_LFC, f)

[nan]


/tmp/ipykernel_3386474/1454611547.py:2: RuntimeWarning: invalid value encountered in divide
  mean_LFC = lfc_profile[:,0]/lfc_profile[:,1]


In [121]:
# #READING BACK IN
# import pickle
# dir2 = dir + f'Project_Algorithms/Domain_Profiles/'
# input_file = dir2 + f'mean_lfc_{res}_{t_res}_{Np_str}.pkl'

# with open(input_file, 'rb') as f:
#     mean_LFC_loaded = pickle.load(f)
# print(mean_LFC_loaded)